In [2]:
# ============================================================
# LSTM WITH ATTENTION - FROM SCRATCH
# Uses: data/train_merged.csv and data/test_merged.csv
# Task: Predict UpDownLabel (0/1) from news titles
# ============================================================

import os
import re
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow version:", tf.__version__)

# ------------------------------------------------------------
# 1. PATHS AND CONSTANTS
# ------------------------------------------------------------
BASE_DIR = "/Users/hibabelhaj/Desktop/LSTMvsTRANSFORMERS"
DATA_DIR = os.path.join(BASE_DIR, "data")
MODELS_DIR = os.path.join(BASE_DIR, "models")

os.makedirs(MODELS_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "train_merged.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test_merged.csv")

VOCAB_SIZE   = 20000        # max number of words to keep in tokenizer
MAX_SEQ_LEN  = 40           # max number of tokens in each title
EMBED_DIM    = 64           # embedding size (learned from scratch)
LSTM_UNITS   = 64           # LSTM hidden size
BATCH_SIZE   = 64
EPOCHS       = 5
RANDOM_STATE = 42

# ------------------------------------------------------------
# 2. LOAD TRAIN AND TEST DATA
# ------------------------------------------------------------
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain sample:")
print(train_df.head())

print("\nTest sample:")
print(test_df.head())

# ------------------------------------------------------------
# 3. BASIC TEXT CLEANING
# ------------------------------------------------------------
def clean_text(text):
    """
    Very simple cleaner:
      - make lowercase
      - remove characters that are not letter, number, or space
      - collapse multiple spaces
    """
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)    # keep letters/digits/space
    text = re.sub(r"\s+", " ", text).strip()    # collapse spaces
    return text

train_df["title_clean"] = train_df["title"].apply(clean_text)
test_df["title_clean"]  = test_df["title"].apply(clean_text)

print("\nExample cleaned title:")
print("RAW :", train_df["title"].iloc[0])
print("CLEAN:", train_df["title_clean"].iloc[0])

# ------------------------------------------------------------
# 4. TOKENIZER AND SEQUENCES
# ------------------------------------------------------------
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<UNK>")
tokenizer.fit_on_texts(train_df["title_clean"].tolist())

# text -> sequence of integers
train_seqs = tokenizer.texts_to_sequences(train_df["title_clean"].tolist())
test_seqs  = tokenizer.texts_to_sequences(test_df["title_clean"].tolist())

# pad sequences to same length
X_train_full = pad_sequences(train_seqs, maxlen=MAX_SEQ_LEN, padding="post", truncating="post")
X_test       = pad_sequences(test_seqs,  maxlen=MAX_SEQ_LEN, padding="post", truncating="post")

y_train_full = train_df["UpDownLabel"].astype(int).values
y_test       = test_df["UpDownLabel"].astype(int).values

print("\nExample sequence:")
print(train_df["title_clean"].iloc[0])
print(X_train_full[0])

print("\nPadded shapes:")
print("X_train_full:", X_train_full.shape)
print("X_test      :", X_test.shape)

# ------------------------------------------------------------
# 5. TRAIN / VALIDATION SPLIT
# ------------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.10,
    random_state=RANDOM_STATE,
    stratify=y_train_full
)

print("\nFinal train shape:", X_train.shape)
print("Validation shape:", X_val.shape)

# ------------------------------------------------------------
# 6. ATTENTION LAYER DEFINITION
# ------------------------------------------------------------
class SimpleAttention(layers.Layer):
    """
    A simple attention layer for sequences.
    Input:  (batch, time, features)
    Output: (batch, features)  -> weighted sum over time
    """
    def __init__(self, **kwargs):
        super(SimpleAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        # input_shape: (batch, time, features)
        self.W = self.add_weight(
            name="att_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True
        )
        self.b = self.add_weight(
            name="att_bias",
            shape=(1,),
            initializer="zeros",
            trainable=True
        )
        super(SimpleAttention, self).build(input_shape)

    def call(self, inputs):
        # inputs: (batch, time, features)
        # 1. Compute "energy" for each time step: tanh(W·h_t + b)
        score = tf.nn.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)  # (batch, time, 1)

        # 2. Convert to attention weights with softmax
        attention_weights = tf.nn.softmax(score, axis=1)  # (batch, time, 1)

        # 3. Weighted sum of inputs using attention weights
        context_vector = tf.reduce_sum(inputs * attention_weights, axis=1)  # (batch, features)

        return context_vector

# ------------------------------------------------------------
# 7. BUILD LSTM + ATTENTION MODEL
# ------------------------------------------------------------
tf.keras.backend.clear_session()

inputs = layers.Input(shape=(MAX_SEQ_LEN,), name="input_tokens")

# Embedding from scratch
x = layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBED_DIM,
    name="embedding"
)(inputs)

# BiLSTM outputs sequence of hidden states (return_sequences=True)
x = layers.Bidirectional(
    layers.LSTM(LSTM_UNITS, return_sequences=True),
    name="bilstm"
)(x)

# Apply attention over the sequence
x = SimpleAttention(name="attention")(x)

# Dense layers for classification
x = layers.Dense(64, activation="relu", name="dense_hidden")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(1, activation="sigmoid", name="output")(x)

model = models.Model(inputs=inputs, outputs=outputs, name="lstm_attention")

model.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

print("\nModel summary:")
model.summary()

# ------------------------------------------------------------
# 8. TRAIN MODEL
# ------------------------------------------------------------
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# ------------------------------------------------------------
# 9. EVALUATE ON TEST SET
# ------------------------------------------------------------
y_prob = model.predict(X_test)
y_pred = (y_prob >= 0.5).astype(int).reshape(-1)

test_acc = accuracy_score(y_test, y_pred)
test_f1  = f1_score(y_test, y_pred)

print("\n=== TEST PERFORMANCE (LSTM + ATTENTION) ===")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test F1 score: {test_f1:.4f}\n")

print("Classification report:")
print(classification_report(y_test, y_pred, digits=4))

# ------------------------------------------------------------
# 10. SAVE MODEL AND TOKENIZER
# ------------------------------------------------------------
model_path = os.path.join(MODELS_DIR, "lstm_attention.h5")
tokenizer_path = os.path.join(MODELS_DIR, "tokenizer_lstm_attention.pkl")

model.save(model_path)
with open(tokenizer_path, "wb") as f:
    pickle.dump(tokenizer, f)

print("\nSaved LSTM+Attention model to:", model_path)
print("Saved tokenizer to:", tokenizer_path)


TensorFlow version: 2.20.0
Train shape: (24090, 8)
Test shape: (5794, 8)

Train sample:
                                               title stock Ticker    NewsDate  \
0  Wells Fargo Maintains Outperform on eBay, Lowe...  EBAY   EBAY  2018-12-21   
1      10 Biggest Price Target Changes For Wednesday  EBAY   EBAY  2018-12-12   
2  Benzinga's Top Upgrades, Downgrades For Decemb...  EBAY   EBAY  2018-12-12   
3  Morgan Stanley Downgrades eBay On Slower GMV G...  EBAY   EBAY  2018-12-12   
4  UPDATE: Morgan Stanley On eBay Downgrade Notes...  EBAY   EBAY  2018-12-12   

   ClosePrice  NextClose  Return1D  UpDownLabel  
0   23.738672  23.345701 -0.016554            0  
1   25.935703  25.917835 -0.000689            0  
2   25.935703  25.917835 -0.000689            0  
3   25.935703  25.917835 -0.000689            0  
4   25.935703  25.917835 -0.000689            0  

Test sample:
                                               title stock Ticker    NewsDate  \
0  BofA Raises Amazon Target O

Model: "lstm_attention"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_tokens (InputLayer)       │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 40, 64)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm (Bidirectional)          │ (None, 40, 128)        │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention (SimpleAttention)     │ (None, 128)            │           129 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,354,498 (5.17 MB)

 Trainable params: 1,354,498 (5.17 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
339/339 ━━━━━━━━━━━━━━━━━━━━ 37s 81ms/step - accuracy: 0.5020 - loss: 0.6934 - val_accuracy: 0.5098 - val_loss: 0.6929
Epoch 2/5
339/339 ━━━━━━━━━━━━━━━━━━━━ 16s 48ms/step - accuracy: 0.5238 - loss: 0.6914 - val_accuracy: 0.5380 - val_loss: 0.6927
Epoch 3/5
339/339 ━━━━━━━━━━━━━━━━━━━━ 15s 43ms/step - accuracy: 0.6070 - loss: 0.6587 - val_accuracy: 0.5189 - val_loss: 0.7018
Epoch 4/5
339/339 ━━━━━━━━━━━━━━━━━━━━ 16s 46ms/step - accuracy: 0.6789 - loss: 0.5824 - val_accuracy: 0.5222 - val_loss: 0.7538
Epoch 5/5
339/339 ━━━━━━━━━━━━━━━━━━━━ 16s 47ms/step - accuracy: 0.7268 - loss: 0.5022 - val_accuracy: 0.5259 - val_loss: 0.8643
182/182 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step



=== TEST PERFORMANCE (LSTM + ATTENTION) ===
Test accuracy: 0.4869
Test F1 score: 0.4665

Classification report:
              precision    recall  f1-score   support

           0     0.4644    0.5551    0.5057      2740
           1     0.5161    0.4257    0.4665      3054

    accuracy                         0.4869      5794
   macro avg     0.4903    0.4904    0.4861      5794
weighted avg     0.4917    0.4869    0.4851      5794


Saved LSTM+Attention model to: /Users/hibabelhaj/Desktop/LSTMvsTRANSFORMERS/models/lstm_attention.h5
Saved tokenizer to: /Users/hibabelhaj/Desktop/LSTMvsTRANSFORMERS/models/tokenizer_lstm_attention.pkl
